# 05 — Impact du Chunking

## Objectif

Étudier l'impact de la taille des chunks et du recouvrement (overlap)
sur la qualité du système RAG.

**Question de recherche :** Quelle combinaison chunk_size × overlap
maximise la qualité des réponses ?

**Paramètres fixes :**
- LLM : Gemini 2.5 Flash
- Embedding : BAAI/bge-m3
- Retrieval : top_k=3
- 3 questions de test

---
**Pourquoi cette comparaison est importante :**
Le chunking est une étape cruciale de l'ingestion. Des chunks mal
dimensionnés peuvent dégrader la qualité du retrieval, quel que soit
le LLM ou l'embedding utilisé.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
import time

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from config import LLMConfig, RetrievalConfig, EmbeddingConfig
from llm_chain import generate_answer
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from langchain_community.vectorstores import FAISS
from embeddings import create_embeddings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

print("✅ Imports réussis")

## 2. Paramètres fixes du pipeline

In [ ]:
llm_cfg = LLMConfig(
    provider="openrouter", model="google/gemini-2.5-flash",
    temperature=0.2, num_predict=512, request_timeout=120,
)
emb_cfg = EmbeddingConfig(provider="huggingface", model="BAAI/bge-m3", device="cpu")
ret_cfg = RetrievalConfig(top_k=3, max_distance=1.5)

print(f"🤖 LLM       : {llm_cfg.provider}:{llm_cfg.model}")
print(f"🔤 Embedding : {emb_cfg.provider}:{emb_cfg.model}")
print(f"📄 Retrieval : top_k={ret_cfg.top_k}")

## 3. Configurations de chunking

On définit les paires (chunk_size, chunk_overlap) à tester.

**Note :** chaque configuration nécessite un index FAISS dédié.
Les indexes doivent avoir été créés au préalable avec `ingest.py`
en modifiant les paramètres CHUNK_SIZE et CHUNK_OVERLAP.

In [ ]:
CHUNK_CONFIGS = [
    ("Baseline (1000/200)",  1000, 200),
    ("Petits (500/100)",     500, 100),
    ("Grands (1500/200)",   1500, 200),
    ("Overlap fort (1000/400)", 1000, 400),
    ("Sans overlap (1000/0)", 1000, 0),
]

# Vérifier quels indexes existent
available = []
for name, size, overlap in CHUNK_CONFIGS:
    slug = f"chunk_{size}_{overlap}"
    if (ROOT / "vectorstore" / slug).exists():
        available.append((name, size, overlap))
        print(f"✅ {name:30s} → index disponible")
    else:
        print(f"❌ {name:30s} → index manquant (créez-le avec ingest.py)")

print(f"\n📋 {len(available)} configuration(s) disponible(s)")

## 4. Questions de test

In [ ]:
QUESTIONS = [
    ("Q01", "Quelle est la note minimale pour valider un module ?",
     "L'étudiant doit obtenir une note finale >= 5,5/10"),
    ("Q02", "Peut-on demander une prolongation du mémoire ?",
     "Oui, avec une demande écrite"),
    ("Q03", "Quel est le montant des frais de prolongation ?",
     "1 000 000 VND par mois"),
]

print(f"📚 {len(QUESTIONS)} questions")

## 5. Juge DeepEval

In [ ]:
judge = create_judge(provider="ollama", model="qwen2.5:3b")
print(f"⚖️  Juge : {judge.get_model_name()}")

## 6. Boucle d'évaluation

Pour chaque configuration de chunking, on charge l'index correspondant
et on exécute le pipeline.

In [ ]:
results = []
embeddings_model = create_embeddings(emb_cfg)

for name, size, overlap in available:
    print(f"\n{'='*60}")
    print(f"  🔄 Test : {name}")
    print(f"{'='*60}")

    slug = f"chunk_{size}_{overlap}"
    index_path = ROOT / "vectorstore" / slug
    db = FAISS.load_local(str(index_path), embeddings_model, allow_dangerous_deserialization=True)

    for qid, question, expected in QUESTIONS::
        try:
        docs_with_scores = db.similarity_search_with_score(question, k=ret_cfg.top_k)
        docs = [d for d, s in docs_with_scores if s <= ret_cfg.max_distance]

        start = time.time()
        answer = generate_answer(question, docs, llm_cfg)
        elapsed = time.time() - start

        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected,
            retrieval_context=[d.page_content for d in docs],
        )

        faith = FaithfulnessMetric(threshold=0.75, model=judge, include_reason=True)
        relev = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)

        try:
            faith.measure(test_case)
        except Exception:
            faith.score = 0.0
        try:
            relev.measure(test_case)
        except Exception:
            relev.score = 0.0

        results.append({
            "Configuration": name,
            "Chunk_Size": size,
            "Chunk_Overlap": overlap,
            "Question": qid,
            "Faithfulness": round(faith.score, 4),
            "AnswerRelevancy": round(relev.score, 4),
            "Temps(s)": round(elapsed, 2),
            "Chunks": len(docs),
        })

        except Exception as e:
            print(f"   [05] ❌ Erreur : {e}")
                    nt(f"   [{qid}] Faith={faith.score:.3f}  Relev={relev.score:.3f}")

df = pd.DataFrame(results)
print(f"\n✅ Terminé : {len(df)} mesures")

## 7. Tableau comparatif

In [ ]:
summary = df.groupby("Configuration")[["Faithfulness", "AnswerRelevancy", "Temps(s)"]].mean().round(4)
summary.columns = ["Fidélité", "Pertinence", "Temps (s)"]
summary

## 8. Graphique

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = df.melt(id_vars=["Configuration"], value_vars=["Faithfulness", "AnswerRelevancy"],
                  var_name="Métrique", value_name="Score")
sns.barplot(data=plot_df, x="Configuration", y="Score", hue="Métrique", ax=ax)
ax.set_title("Impact du Chunking", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 9. Export CSV

In [ ]:
out_dir = ROOT / "evaluation" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "05_compare_chunking.csv", index=False)
summary.to_csv(out_dir / "05_compare_chunking_summary.csv")
print(f"✅ Exporté dans {out_dir}/")

## 10. Analyse

**Lecture des résultats :**
- **Petits chunks** : meilleure granularité mais risque de perdre le contexte
- **Grands chunks** : plus de contexte mais risque de mélanger les sujets
- **Overlap fort** : évite de couper des informations importantes
- **Sans overlap** : économique mais peut perdre des passages charnières

**Recommandation :** un chunk_size de 1000 avec overlap de 200 est
un bon point de départ. Ajustez selon la nature de vos documents.